# Calibration Curves by Subject

Visualize isotonic vs. kernel-smoothed reliability curves. Identify subjects where the monotonicity assumption diverges from the unconstrained fit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL = 'gpt4o'  # or 'llama'
curves = pd.read_parquet(f'../data/processed/calibration_curves_{MODEL}.parquet')
summaries = pd.read_parquet(f'../data/processed/subject_summaries_{MODEL}.parquet')
summaries.head(10)

In [ ]:
def plot_curve(row, ax):
    bins = np.array(row['confidence_bins'])
    ax.plot([0,1],[0,1], 'k--', lw=1, label='Perfect')
    ax.plot(bins, row['isotonic_acc'], label='Isotonic')
    ax.plot(bins, row['kernel_acc'], label='Kernel', linestyle=':')
    ax.set_title(f"{row['subject']}\nECE={row['ece']:.3f}")
    ax.legend(fontsize=7)

# Plot top-5 most miscalibrated subjects
top5 = curves.merge(summaries[['subject','ece']], on='subject').nlargest(5, 'ece')
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, (_, row) in zip(axes, top5.iterrows()):
    plot_curve(row, ax)
plt.tight_layout()
plt.savefig(f'../report/figures/top5_miscalibrated_{MODEL}.pdf', bbox_inches='tight')